# Assignment 1A — Medical & Clinical Literature

This notebook implements Part A end to end: PDF extraction and cleaning, document-level held-out splitting, tokenizer-aligned BOS/EOS packing, model inspection, continual pre-training (CPT), perplexity evaluation, and a catastrophic-forgetting check.

The notebook is designed to run from this repository root with the local `.venv` kernel. It reports the size of the PDF-derived corpus and warns when the cleaned text is below the assignment's recommended 10–50 MB collection target.

In [ ]:
# Run this notebook with: .venv/bin/jupyter lab
# If dependencies are missing, install them into the local environment:
# %pip install -r requirements.txt

from __future__ import annotations

import json
import logging
import math
import os
import random
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Resolve the project root whether Jupyter was opened here or one directory above.
ROOT = Path.cwd()
RAW_PDFS = ROOT / 'raw_pdfs'
EXTRACTED = ROOT / 'extracted_text'
CLEAN_CORPUS = ROOT / 'clean_corpus'
OUTPUTS = ROOT / 'outputs'
for folder in (EXTRACTED, CLEAN_CORPUS, OUTPUTS):
    folder.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'microsoft/biogpt-large'
SEQUENCE_LENGTH = 1024  # BioGPT context length; change only with a documented model choice.
CPT_DIR = OUTPUTS / 'biogpt-large-cpt'
PACKED_PARQUET = OUTPUTS / 'medical_packed_train.parquet'
BASELINE_PATH = OUTPUTS / 'baseline_outputs.json'
AUDIT_PATH = OUTPUTS / 'model_audit.json'
LOSS_HISTORY_PATH = OUTPUTS / 'cpt_loss_history.json'
LOSS_PLOT_PATH = OUTPUTS / 'cpt_loss_curve.png'
EVAL_PATH = OUTPUTS / 'evaluation_results.json'
SPLIT_MANIFEST = OUTPUTS / 'corpus_split.json'

# Data preparation is safe to run locally. Model download/training is intentionally opt-in.
RUN_TOKENIZATION = True
RUN_MODEL_INSPECTION = True
RUN_CPT = True
RUN_EVALUATION = True

def detect_runtime():
    import torch
    if torch.cuda.is_available():
        bf16 = getattr(torch.cuda, 'is_bf16_supported', lambda: False)()
        return {'device': 'cuda', 'dtype': 'bfloat16' if bf16 else 'float16', 'cuda': True, 'mps': False}
    mps = getattr(torch.backends, 'mps', None)
    if mps is not None and mps.is_available():
        return {'device': 'mps', 'dtype': 'float32', 'cuda': False, 'mps': True}
    return {'device': 'cpu', 'dtype': 'float32', 'cuda': False, 'mps': False}

print({'python': sys.executable, 'root': str(ROOT), 'model': MODEL_ID, 'runtime': detect_runtime()})

{'python': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/.venv/bin/python', 'root': '/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution', 'model': 'microsoft/biogpt-large', 'runtime': {'device': 'mps', 'dtype': 'float32', 'cuda': False, 'mps': True}}


In [ ]:
# Pipeline control flags (sequential execution)
import certifi
import httpx
import transformers.tokenization_utils_base as tokenization_utils_base
from huggingface_hub import set_async_client_factory, set_client_factory
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'
os.environ['HF_HUB_DISABLE_XET'] = '1'

def _hf_client_factory():
    return httpx.Client(verify=False, timeout=60)

def _hf_async_client_factory():
    return httpx.AsyncClient(verify=False, timeout=60)

set_client_factory(_hf_client_factory)
set_async_client_factory(_hf_async_client_factory)
tokenization_utils_base.list_repo_templates = lambda *args, **kwargs: []

RUN_TOKENIZATION = True
RUN_MODEL_INSPECTION = True
RUN_CPT = True
RUN_EVALUATION = True
print({'HF_HUB_DISABLE_SSL_VERIFY': os.environ['HF_HUB_DISABLE_SSL_VERIFY'], 'HF_HUB_DISABLE_XET': os.environ['HF_HUB_DISABLE_XET'], 'RUN_TOKENIZATION': RUN_TOKENIZATION, 'RUN_MODEL_INSPECTION': RUN_MODEL_INSPECTION, 'RUN_CPT': RUN_CPT, 'RUN_EVALUATION': RUN_EVALUATION})

/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'HF_HUB_DISABLE_SSL_VERIFY': '1', 'HF_HUB_DISABLE_XET': '1', 'RUN_TOKENIZATION': True, 'RUN_MODEL_INSPECTION': True, 'RUN_CPT': False, 'RUN_EVALUATION': False}


## Step 1 — Data collection, extraction, and cleaning

Each PDF is extracted page-by-page into one text file. Cleaning is applied in the required order: length, within-document paragraph repetition, exact-document deduplication, then English-language retention. Counts and the largest reduction are saved for reporting.

In [3]:
class PdfWarningCollector(logging.Handler):
    """Collect recoverable pypdf warnings without flooding notebook output."""

    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.messages = {}

    def emit(self, record):
        message = record.getMessage()
        normalized = re.sub(r' at byte 0x[0-9a-f]+', ' at byte <offset>', message)
        self.messages[normalized] = self.messages.get(normalized, 0) + 1


def configure_pdf_warning_capture():
    collector = PdfWarningCollector()
    collector._assignment_pdf_handler = True
    pypdf_logger = logging.getLogger('pypdf')
    for handler in list(pypdf_logger.handlers):
        if getattr(handler, '_assignment_pdf_handler', False):
            pypdf_logger.removeHandler(handler)
    pypdf_logger.addHandler(collector)
    pypdf_logger.setLevel(logging.WARNING)
    pypdf_logger.propagate = False
    return collector

def extract_pdfs_page_by_page(pdf_dir, extracted_dir):
    from pypdf import PdfReader
    warning_collector = configure_pdf_warning_capture()
    extracted_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for pdf_path in sorted(pdf_dir.glob('*.pdf')):
        reader = PdfReader(str(pdf_path), strict=False)
        pages = []
        for page_number, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or '').replace('\x00', '').strip()
            pages.append(f'[PAGE {page_number}]\n{text}')
        text_path = extracted_dir / f'{pdf_path.stem}.txt'
        text_path.write_text('\n\n'.join(pages), encoding='utf-8')
        rows.append({'file': pdf_path.name, 'pages': len(pages), 'characters': sum(len(p) for p in pages)})
    warning_report = {
        'warning_count': sum(warning_collector.messages.values()),
        'by_message': warning_collector.messages,
        'note': 'These are recoverable non-compliance warnings handled by pypdf with strict=False; extraction completed.'
    }
    (OUTPUTS / 'pdf_parser_warnings.json').write_text(json.dumps(warning_report, indent=2), encoding='utf-8')
    if warning_report['warning_count']:
        print({'pdf_parser_warnings': warning_report['warning_count'], 'report': str(OUTPUTS / 'pdf_parser_warnings.json')})
    return rows

def paragraphs(text):
    blocks = re.split(r'\n\s*\n+', text)
    return [re.sub(r'\s+', ' ', block).strip() for block in blocks if block.strip()]

def looks_english(text):
    # Lightweight, dependency-free language filter suitable for this notebook.
    words = re.findall(r'[A-Za-z]+', text.lower())
    stopwords = {'the', 'and', 'of', 'to', 'in', 'is', 'for', 'with', 'on', 'as', 'by', 'this', 'an', 'or', 'from', 'are', 'be', 'can', 'which', 'when'}
    if not words:
        return False
    ascii_ratio = sum(ord(ch) < 128 for ch in text) / max(len(text), 1)
    stopword_hits = sum(word in stopwords for word in words)
    return ascii_ratio >= 0.85 and stopword_hits >= max(2, min(10, len(words) // 30))

def clean_extracted_text(extracted_dir, cleaned_dir, min_chars=50, duplicate_fraction=0.30):
    files = sorted(extracted_dir.glob('*.txt'))
    counts = {'before': len(files), 'after_length': 0, 'after_repetition': 0, 'after_deduplication': 0, 'after_language': 0}
    candidates = []
    for path in files:
        text = path.read_text(encoding='utf-8', errors='ignore').strip()
        if len(text) >= min_chars:
            candidates.append((path, text))
    counts['after_length'] = len(candidates)

    non_repetitive = []
    for path, text in candidates:
        paras = paragraphs(text)
        duplicate_ratio = (len(paras) - len(set(paras))) / max(len(paras), 1)
        if duplicate_ratio <= duplicate_fraction:
            non_repetitive.append((path, text))
    counts['after_repetition'] = len(non_repetitive)

    seen = set()
    unique = []
    for path, text in non_repetitive:
        key = re.sub(r'\s+', ' ', text).strip()
        if key not in seen:
            seen.add(key)
            unique.append((path, text))
    counts['after_deduplication'] = len(unique)

    final = [(path, text) for path, text in unique if looks_english(text)]
    counts['after_language'] = len(final)
    cleaned_dir.mkdir(parents=True, exist_ok=True)
    for old_path in cleaned_dir.glob('*.txt'):
        old_path.unlink()
    for path, text in final:
        (cleaned_dir / path.name).write_text(text, encoding='utf-8')

    impact = {
        'length_filter': counts['before'] - counts['after_length'],
        'repetition_filter': counts['after_length'] - counts['after_repetition'],
        'deduplication': counts['after_repetition'] - counts['after_deduplication'],
        'language_filter': counts['after_deduplication'] - counts['after_language'],
    }
    greatest_impact = max(impact, key=impact.get) if max(impact.values(), default=0) > 0 else 'none (no documents removed)'
    return counts, impact, greatest_impact

if not list(RAW_PDFS.glob('*.pdf')):
    raise FileNotFoundError(f'No PDFs found in {RAW_PDFS}. Add licensed/open-access medical PDFs before running Step 1.')

extraction_stats = extract_pdfs_page_by_page(RAW_PDFS, EXTRACTED)
clean_counts, clean_impact, greatest_impact = clean_extracted_text(EXTRACTED, CLEAN_CORPUS)
corpus_bytes = sum(p.stat().st_size for p in CLEAN_CORPUS.glob('*.txt'))
summary = {
    'pdf_documents': len(extraction_stats),
    'pages_extracted': sum(row['pages'] for row in extraction_stats),
    'raw_extracted_characters': sum(row['characters'] for row in extraction_stats),
    'clean_corpus_bytes': corpus_bytes,
    'clean_corpus_megabytes': round(corpus_bytes / 1024**2, 4),
    'cleaning_counts': clean_counts,
    'removed_by_step': clean_impact,
    'greatest_impact': greatest_impact,
}
if corpus_bytes < 10 * 1024**2:
    warnings.warn('The clean corpus is below the recommended 10 MB minimum; add more domain PDFs for a marked run.')
print(json.dumps(summary, indent=2))
(OUTPUTS / 'step1_cleaning_report.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

{
  "pdf_documents": 12,
  "pages_extracted": 12,
  "raw_extracted_characters": 14817,
  "clean_corpus_bytes": 14841,
  "clean_corpus_megabytes": 0.0142,
  "cleaning_counts": {
    "before": 12,
    "after_length": 12,
    "after_repetition": 12,
    "after_deduplication": 12,
    "after_language": 12
  },
  "removed_by_step": {
    "length_filter": 0,
    "repetition_filter": 0,
    "deduplication": 0,
    "language_filter": 0
  },
  "greatest_impact": "none (no documents removed)"
}


/var/folders/8r/xb4x4vnd7f17ndy4g8yn7xbw0000gn/T/ipykernel_6141/3962894986.py:91: UserWarning: The clean corpus is below the recommended 10 MB minimum; add more domain PDFs for a marked run.
  warnings.warn('The clean corpus is below the recommended 10 MB minimum; add more domain PDFs for a marked run.')


489

## Held-out split

Ten percent of documents is reserved at the document level before tokenization. Only training documents are packed for CPT; the held-out documents are used exclusively for the Step 5 perplexity comparison.

In [4]:
def split_corpus_files(cleaned_dir, manifest_path, eval_fraction=0.10):
    files = sorted(cleaned_dir.glob('*.txt'))
    if len(files) < 2:
        raise ValueError('At least two cleaned documents are required for a held-out split.')
    rng = random.Random(SEED)
    shuffled = files.copy()
    rng.shuffle(shuffled)
    eval_count = max(1, math.ceil(len(shuffled) * eval_fraction))
    eval_files = sorted(shuffled[:eval_count])
    train_files = sorted(shuffled[eval_count:])
    manifest = {'seed': SEED, 'eval_fraction': eval_fraction, 'train_files': [p.name for p in train_files], 'eval_files': [p.name for p in eval_files]}
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return train_files, eval_files, manifest

train_files, eval_files, split_manifest = split_corpus_files(CLEAN_CORPUS, SPLIT_MANIFEST)
train_texts = [p.read_text(encoding='utf-8') for p in train_files]
eval_texts = [p.read_text(encoding='utf-8') for p in eval_files]
print({'train_documents': len(train_files), 'eval_documents': len(eval_files), 'train_files': [p.name for p in train_files], 'eval_files': [p.name for p in eval_files]})

{'train_documents': 10, 'eval_documents': 2, 'train_files': ['medical_topic_01.txt', 'medical_topic_02.txt', 'medical_topic_03.txt', 'medical_topic_04.txt', 'medical_topic_05.txt', 'medical_topic_07.txt', 'medical_topic_09.txt', 'medical_topic_10.txt', 'medical_topic_11.txt', 'medical_topic_12.txt'], 'eval_files': ['medical_topic_06.txt', 'medical_topic_08.txt']}


## Step 2 — Tokenization and packed dataset

The selected model's own tokenizer is loaded. Text is converted to token IDs through the tokenizer's low-level tokenize/ID conversion path so a long document is never prepared as one oversized model input. Every training document receives BOS and EOS markers, documents are concatenated into one stream, and the stream is sliced into fixed-length sequences without padding. The packed training set is saved as Parquet.

In [5]:
def encode_without_model_limit(tokenizer, text):
    """Convert text to IDs without asking Transformers to prepare one huge model input."""
    tokens = tokenizer.tokenize(text)
    return tokenizer.convert_tokens_to_ids(tokens)

def tokenize_and_pack(files, parquet_path, model_id, sequence_length):
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    bos_id, eos_id = tokenizer.bos_token_id, tokenizer.eos_token_id
    if bos_id is None:
        bos_id = eos_id if eos_id is not None else tokenizer.unk_token_id
        warnings.warn('Tokenizer has no BOS token; using EOS/UNK fallback.')
    if eos_id is None:
        eos_id = bos_id
        warnings.warn('Tokenizer has no EOS token; using BOS fallback.')
    if bos_id is None or eos_id is None:
        raise ValueError('The selected tokenizer has no usable special token IDs.')

    stream, document_lengths = [], []
    for path in files:
        ids = encode_without_model_limit(tokenizer, path.read_text(encoding='utf-8'))
        ids = [bos_id] + ids + [eos_id]
        stream.extend(ids)
        document_lengths.append(len(ids))
    sequence_count = len(stream) // sequence_length
    if sequence_count == 0:
        raise ValueError('No complete packed sequence was produced; add more text or reduce sequence_length.')
    rows = [{'input_ids': stream[i:i + sequence_length], 'attention_mask': [1] * sequence_length} for i in range(0, sequence_count * sequence_length, sequence_length)]
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_parquet(parquet_path, index=False)
    return tokenizer, {
        'tokenizer': model_id,
        'documents_packed': len(files),
        'total_tokens_in_stream': len(stream),
        'average_document_length_tokens': round(sum(document_lengths) / len(document_lengths), 2),
        'packed_sequences': sequence_count,
        'sequence_length': sequence_length,
        'unused_remainder_tokens': len(stream) % sequence_length,
        'path': str(parquet_path),
    }

if RUN_TOKENIZATION:
    tokenizer, packing_stats = tokenize_and_pack(train_files, PACKED_PARQUET, MODEL_ID, SEQUENCE_LENGTH)
    print(json.dumps(packing_stats, indent=2))
else:
    print('Tokenization is implemented but disabled. Set RUN_TOKENIZATION=True to download the model tokenizer and write Parquet.')

{
  "tokenizer": "microsoft/biogpt-large",
  "documents_packed": 10,
  "total_tokens_in_stream": 2057,
  "average_document_length_tokens": 205.7,
  "packed_sequences": 2,
  "sequence_length": 1024,
  "unused_remainder_tokens": 9,
  "path": "/Users/madhg/ws/Jupyter/Notebooks/LLM/LLM_Assignment_Medical_Solution/outputs/medical_packed_train.parquet"
}


## Step 3 — Model loading, architecture audit, and baseline

The model is loaded with `AutoModelForCausalLM.from_pretrained`. Precision follows the detected hardware, gradient checkpointing is enabled for training, and the audit verifies the decoder depth, attention geometry, parameter counts, and vocabulary/output-head alignment.

In [6]:
def load_model_and_audit(model_id, enable_gradient_checkpointing=True):
    import torch
    from transformers import AutoConfig, AutoModelForCausalLM
    runtime = detect_runtime()
    config = AutoConfig.from_pretrained(model_id)
    dtype = torch.float32
    kwargs = {}
    if runtime['device'] == 'cuda':
        dtype = torch.bfloat16 if runtime['dtype'] == 'bfloat16' else torch.float16
        kwargs.update(torch_dtype=dtype, device_map='auto')
    else:
        kwargs['torch_dtype'] = dtype
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if runtime['device'] in {'cpu', 'mps'}:
        model = model.to(runtime['device'])
    if enable_gradient_checkpointing and hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()
        if hasattr(model, 'config'):
            model.config.use_cache = False
    layers = getattr(config, 'num_hidden_layers', getattr(config, 'n_layer', None))
    heads = getattr(config, 'num_attention_heads', getattr(config, 'n_head', None))
    hidden = getattr(config, 'hidden_size', getattr(config, 'n_embd', None))
    head_dim = getattr(config, 'head_dim', None) or (hidden // heads if hidden and heads else None)
    lm_head = getattr(model, 'lm_head', None)
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    audit = {'model_id': model_id, 'device': runtime['device'], 'dtype': str(dtype), 'total_parameters': total, 'trainable_parameters': trainable, 'decoder_layers': layers, 'attention_heads': heads, 'hidden_size': hidden, 'head_dimension': head_dim, 'vocab_size': config.vocab_size, 'lm_head_output_dimension': getattr(lm_head, 'out_features', None), 'lm_head_matches_vocab': getattr(lm_head, 'out_features', None) == config.vocab_size}
    return model, config, audit

def generate_outputs(model, tokenizer, prompts, max_new_tokens=60):
    import torch
    model.eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    device = next(model.parameters()).device
    results = []
    for prompt in prompts:
        inputs = {key: value.to(device) for key, value in tokenizer(prompt, return_tensors='pt').items()}
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        results.append({'prompt': prompt, 'generated_text': tokenizer.decode(generated[0], skip_special_tokens=True)})
    return results

DOMAIN_PROMPTS = ['Explain why repeated blood-pressure measurements are useful in hypertension.', 'What is the relationship between insulin resistance and type 2 diabetes?', 'Why is antimicrobial stewardship important?']

if RUN_MODEL_INSPECTION:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    base_model, model_config, audit = load_model_and_audit(MODEL_ID)
    baseline = generate_outputs(base_model, tokenizer, DOMAIN_PROMPTS)
    print(json.dumps(audit, indent=2))
    AUDIT_PATH.write_text(json.dumps(audit, indent=2), encoding='utf-8')
    BASELINE_PATH.write_text(json.dumps(baseline, indent=2), encoding='utf-8')
    baseline
else:
    print('Model inspection is implemented but disabled. Set RUN_MODEL_INSPECTION=True to download and load BioGPT-Large.')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


OSError: Can't load the model for 'microsoft/biogpt-large'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/biogpt-large' is the correct path to a directory containing a file named pytorch_model.bin.

## Step 4 — CPT training loop and loss analysis

CPT uses the packed training-only Parquet data, Hugging Face `Trainer`, AdamW, linear warmup, and a custom callback that records every logging-step loss. The history and loss plot are saved under `outputs/`.

In [ ]:
class PackedTextDataset:
    def __new__(cls, parquet_path):
        import torch
        frame = pd.read_parquet(parquet_path)
        class Dataset(torch.utils.data.Dataset):
            def __len__(self):
                return len(frame)
            def __getitem__(self, index):
                return {'input_ids': torch.tensor(frame.iloc[index]['input_ids'], dtype=torch.long), 'attention_mask': torch.tensor(frame.iloc[index]['attention_mask'], dtype=torch.long)}
        return Dataset()

def train_cpt(model, tokenizer, dataset, output_dir, max_steps=100, learning_rate=5e-5, warmup_steps=10, batch_size=1):
    import torch
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainerCallback, TrainingArguments
    class LossHistoryCallback(TrainerCallback):
        def __init__(self):
            self.history = []
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and 'loss' in logs:
                self.history.append({'step': int(state.global_step), 'loss': float(logs['loss'])})
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    runtime = detect_runtime()
    callback = LossHistoryCallback()
    args = TrainingArguments(output_dir=str(output_dir), max_steps=max_steps, learning_rate=learning_rate, warmup_steps=warmup_steps, lr_scheduler_type='linear', optim='adamw_torch', per_device_train_batch_size=batch_size, gradient_accumulation_steps=8, logging_steps=1, save_steps=max_steps, save_total_limit=1, fp16=runtime['device'] == 'cuda' and runtime['dtype'] == 'float16', bf16=runtime['device'] == 'cuda' and runtime['dtype'] == 'bfloat16', report_to='none', remove_unused_columns=False)
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False), callbacks=[callback])
    trainer.train()
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return callback.history

def plot_loss_history(history, output_path):
    import matplotlib.pyplot as plt
    if not history:
        warnings.warn('No loss records were captured; no loss curve was written.')
        return
    frame = pd.DataFrame(history)
    ax = frame.plot(x='step', y='loss', marker='o', title='CPT training loss', legend=False, figsize=(8, 4))
    ax.set_xlabel('Training step')
    ax.set_ylabel('Loss')
    ax.grid(alpha=0.25)
    ax.figure.tight_layout()
    ax.figure.savefig(output_path, dpi=160)
    plt.show()

if RUN_CPT:
    if not PACKED_PARQUET.exists():
        raise FileNotFoundError('Run Step 2 first to create the packed training Parquet file.')
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    cpt_model, _, _ = load_model_and_audit(MODEL_ID)
    history = train_cpt(cpt_model, tokenizer, PackedTextDataset(PACKED_PARQUET), CPT_DIR)
    LOSS_HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding='utf-8')
    plot_loss_history(history, LOSS_PLOT_PATH)
    print({'initial_logged_loss': history[0]['loss'] if history else None, 'final_logged_loss': history[-1]['loss'] if history else None, 'checkpoint': str(CPT_DIR)})
else:
    print('CPT training is implemented but disabled. Enable Steps 2–4 and set RUN_CPT=True for a GPU-backed run.')

CPT training is implemented but disabled. Enable Steps 2–4 and set RUN_CPT=True for a GPU-backed run.


## Step 5 — Perplexity and catastrophic forgetting

Perplexity is computed with shifted causal cross-entropy over the same held-out document split for the base and CPT models. General-domain generations are saved side-by-side with a transparent keyword-based verdict heuristic; the verdict is a screening aid and should be reviewed qualitatively.

In [ ]:
def perplexity(model, tokenizer, texts, max_length=1024):
    import torch
    model.eval()
    total_nll, total_tokens = 0.0, 0
    device = next(model.parameters()).device
    for text in texts:
        ids = encode_without_model_limit(tokenizer, text)
        for start in range(0, len(ids), max_length):
            chunk = ids[start:start + max_length]
            if len(chunk) < 2:
                continue
            input_ids = torch.tensor([chunk], dtype=torch.long, device=device)
            with torch.no_grad():
                logits = model(input_ids=input_ids).logits[:, :-1, :]
            targets = input_ids[:, 1:]
            loss_sum = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction='sum')
            total_nll += float(loss_sum.item())
            total_tokens += targets.numel()
    mean_nll = total_nll / max(total_tokens, 1)
    return {'mean_nll': mean_nll, 'perplexity': math.exp(min(mean_nll, 20)), 'tokens': total_tokens}

def forgetting_verdict(base_text, cpt_text, expected_terms):
    base = base_text.lower()
    cpt = cpt_text.lower()
    base_hits = sum(term.lower() in base for term in expected_terms)
    cpt_hits = sum(term.lower() in cpt for term in expected_terms)
    return 'Degraded' if base_hits > 0 and cpt_hits == 0 else 'Retained'

GENERAL_PROMPTS = ['The capital of France is', 'Water boils at', 'The speed of light is approximately']
EXPECTED_TERMS = [['Paris'], ['100', 'one hundred'], ['299', '300,000', 'three hundred']]

if RUN_EVALUATION:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    base_model, _, _ = load_model_and_audit(MODEL_ID, enable_gradient_checkpointing=False)
    cpt_path = CPT_DIR if CPT_DIR.exists() else None
    if cpt_path is None:
        raise FileNotFoundError('CPT checkpoint not found. Run Step 4 before Step 5.')
    cpt_model, _, _ = load_model_and_audit(str(cpt_path), enable_gradient_checkpointing=False)
    base_ppl = perplexity(base_model, tokenizer, eval_texts, SEQUENCE_LENGTH)
    cpt_ppl = perplexity(cpt_model, tokenizer, eval_texts, SEQUENCE_LENGTH)
    reduction = 100 * (base_ppl['perplexity'] - cpt_ppl['perplexity']) / max(base_ppl['perplexity'], 1e-12)
    base_general = generate_outputs(base_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    cpt_general = generate_outputs(cpt_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    comparison = []
    for index, prompt in enumerate(GENERAL_PROMPTS):
        comparison.append({'prompt': prompt, 'base_output': base_general[index]['generated_text'], 'cpt_output': cpt_general[index]['generated_text'], 'verdict': forgetting_verdict(base_general[index]['generated_text'], cpt_general[index]['generated_text'], EXPECTED_TERMS[index])})
    results = {'base_perplexity': base_ppl, 'cpt_perplexity': cpt_ppl, 'ppl_reduction_percent': reduction, 'general_domain_comparison': comparison}
    EVAL_PATH.write_text(json.dumps(results, indent=2), encoding='utf-8')
    print(json.dumps(results, indent=2))
else:
    print('Evaluation is implemented but disabled. Enable it after a completed CPT checkpoint exists.')

Evaluation is implemented but disabled. Enable it after a completed CPT checkpoint exists.


## Deliverables produced

- `extracted_text/*.txt`: page-wise PDF extraction
- `clean_corpus/*.txt`: cleaned English corpus
- `outputs/step1_cleaning_report.json`: counts and largest cleaning impact
- `outputs/corpus_split.json`: deterministic 90/10 document split
- `outputs/medical_packed_train.parquet`: tokenizer-aligned CPT sequences
- `outputs/biogpt-large-cpt/`: final CPT checkpoint and tokenizer
- `outputs/cpt_loss_curve.png` and `outputs/cpt_loss_history.json`: training loss analysis
- `outputs/evaluation_results.json`: domain PPL reduction and forgetting comparison

For the final run, set `RUN_TOKENIZATION`, `RUN_MODEL_INSPECTION`, `RUN_CPT`, and `RUN_EVALUATION` to `True` in order, and use a runtime with enough memory for the selected model.